# Random Forest

Learn this model in order: **setup → baseline → initial model → tuning → interpretation**.

We keep all 395 students, including 38 recorded zero grades. The same 316 training students and five folds are used in every notebook. The 79 test students belong to final evaluation in Notebook 04.

There are two clearly separated scenarios: without G1/G2 (main) and with both grades (later stage). Input timing, particularly absences, still needs verification. MAE remains the selection metric. The test results have already been inspected: rerunning the unchanged experiment reproduces it, rather than creating a new unseen test.

Results and explanations stay in this notebook. Only fitted pipelines are saved to `models/` for the comparison and evaluation notebooks.

## 1 Setup

Shared helpers in `src/model_utils.py` handle the split and repeated metric calculations. Model settings, fitting and tuning remain visible here. Preprocessing is always fitted inside the training fold.

MAE/RMSE measure grade-point error; R² describes squared-error performance. Accuracy, precision, recall, F1 and specificity evaluate the derived **predicted G3 < 10** flag against **actual G3 < 10**. No rounding or threshold tuning is used.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate, GridSearchCV

PROJECT_ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data/student-mat.csv").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from model_utils import (
    load_data, model_pipeline, preprocessing_for,
    CV, SCORING, METRICS, fold_results, cv_summary, pooled_risk,
)

df, train_ids, test_ids = load_data(PROJECT_ROOT)
y_train = df.loc[train_ids, "G3"]
X_no_grades = df.loc[train_ids].drop(columns=["G1", "G2", "G3"])
X_with_grades = df.loc[train_ids].drop(columns="G3")
print("Training:", len(train_ids), "| Held-out test:", len(test_ids))
print("Inputs without / with G1,G2:", X_no_grades.shape[1], "/", X_with_grades.shape[1])

Training: 316 | Held-out test: 79
Inputs without / with G1,G2: 30 / 32


In [2]:
MODEL_NAME = 'Random Forest'
MODEL_SLUG = 'random_forest'
SCALE_NUMERIC = False

from sklearn.ensemble import RandomForestRegressor
INITIAL_ESTIMATOR = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.8, random_state=42, n_jobs=1)
MODEL_PARAM_GRID = {'model__max_depth': [None, 5, 10], 'model__min_samples_leaf': [2, 5, 10], 'model__max_features': [0.8, 1.0]}

## 2 Constant baselines

Mean and median baselines learn their constant from each fold's training labels. They ignore the inputs, so one set of scores applies to both scenarios. Precision is undefined when nobody is flagged; missing values are left visible rather than replaced with perfect scores.

In [3]:
baseline_folds = {}
for strategy in ["mean", "median"]:
    scores = cross_validate(DummyRegressor(strategy=strategy), X_no_grades, y_train,
        cv=CV, scoring=SCORING, error_score="raise")
    baseline_folds[strategy] = fold_results(scores)

display(pd.DataFrame({name: folds[METRICS].mean()
    for name, folds in baseline_folds.items()}).T.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.4029,4.5635,-0.0350,0.6744,NaN,0.0,0.0,1.0
median,3.3403,4.5931,-0.0519,0.6744,NaN,0.0,0.0,1.0


## 3 Without G1 and G2

This is the main prediction scenario. Excluding grades does not by itself prove that all other inputs are available early.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [4]:
initial_no_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=False)
initial_scores_no_grades = cross_validate(initial_no_grades, X_no_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_no_grades = fold_results(initial_scores_no_grades)
display(cv_summary(initial_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.0015,3.9525,0.2051,0.6868,0.5263,0.4434,0.4687,0.8100
std,0.3680,0.3494,0.1333,0.0477,0.1422,0.1484,0.1072,0.0689
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [5]:
grid_no_grades = dict(MODEL_PARAM_GRID)
grid_no_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=False)
    for use_log in [False, True]
]
search_no_grades = GridSearchCV(initial_no_grades, grid_no_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_no_grades.fit(X_no_grades, y_train)
best_no_grades = search_no_grades.best_estimator_
tuned_folds_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_)

print("Model settings:", best_no_grades.named_steps["model"])
print("Log absences:", best_no_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,2.9641,3.9366,0.2081,0.6932,0.5333,0.4939,0.5050,0.7950
std,0.3456,0.3204,0.1538,0.0351,0.0889,0.0984,0.0575,0.0386
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: RandomForestRegressor(max_features=0.8, min_samples_leaf=2, n_estimators=300,
                      n_jobs=1, random_state=42)
Log absences: False


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [6]:
comparison_no_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_no_grades[METRICS].mean(),
    "Tuned model": tuned_folds_no_grades[METRICS].mean(),
}).T
display(comparison_no_grades.round(4))

training_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_no_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_no_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_no_grades),
    "Tuned model": pooled_risk(tuned_folds_no_grades)}).T.round(4))
display(tuned_folds_no_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.0000,1.000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.0000,1.000
Initial model,3.0015,3.9525,0.2051,0.6868,0.5263,0.4434,0.4687,0.810
Tuned model,2.9641,3.9366,0.2081,0.6932,0.5333,0.4939,0.5050,0.795


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.6867,0.5233,0.4369,0.4762,0.8075,45.0,41.0,58.0,172.0
Tuned model,0.6930,0.5319,0.4854,0.5076,0.7934,50.0,44.0,53.0,169.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,2.5595,3.4942,0.1497,0.6562,0.6667,0.4286,0.5217,0.8333,30.0,6.0,16.0,12.0
1,2,3.4765,4.3389,0.3748,0.6984,0.5455,0.5714,0.5581,0.7619,32.0,10.0,9.0,12.0
2,3,2.7512,3.7744,-0.0143,0.6667,0.4211,0.4444,0.4324,0.7556,34.0,11.0,10.0,8.0
3,4,3.0352,3.9817,0.2038,0.7460,0.5000,0.6250,0.5556,0.7872,37.0,10.0,6.0,10.0
4,5,2.9979,4.0940,0.3263,0.6984,0.5333,0.4000,0.4571,0.8372,36.0,7.0,12.0,8.0


Tuned training MAE: 1.3753
Tuned validation MAE: 2.9641


## 4 With G1 and G2

This is a later-stage comparison, usable only once both prior grades are known. It answers a different prediction-timing question.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [7]:
initial_with_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=True)
initial_scores_with_grades = cross_validate(initial_with_grades, X_with_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_with_grades = fold_results(initial_scores_with_grades)
display(cv_summary(initial_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,0.9846,1.5559,0.8807,0.8797,0.7288,0.9889,0.8376,0.8262
std,0.2231,0.4584,0.0353,0.0311,0.0776,0.0248,0.0573,0.0346
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [8]:
grid_with_grades = dict(MODEL_PARAM_GRID)
grid_with_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=True)
    for use_log in [False, True]
]
search_with_grades = GridSearchCV(initial_with_grades, grid_with_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_with_grades.fit(X_with_grades, y_train)
best_with_grades = search_with_grades.best_estimator_
tuned_folds_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_)

print("Model settings:", best_with_grades.named_steps["model"])
print("Log absences:", best_with_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,0.9381,1.4757,0.8928,0.8703,0.7149,0.9889,0.8287,0.8104
std,0.2056,0.4797,0.0369,0.0303,0.0679,0.0248,0.0507,0.0478
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: RandomForestRegressor(max_features=0.8, min_samples_leaf=2, n_estimators=300,
                      n_jobs=1, random_state=42)
Log absences: True


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [9]:
comparison_with_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_with_grades[METRICS].mean(),
    "Tuned model": tuned_folds_with_grades[METRICS].mean(),
}).T
display(comparison_with_grades.round(4))

training_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_with_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_with_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_with_grades),
    "Tuned model": pooled_risk(tuned_folds_with_grades)}).T.round(4))
display(tuned_folds_with_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.0000,1.0000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.0000,1.0000
Initial model,0.9846,1.5559,0.8807,0.8797,0.7288,0.9889,0.8376,0.8262
Tuned model,0.9381,1.4757,0.8928,0.8703,0.7149,0.9889,0.8287,0.8104


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.8797,0.7338,0.9903,0.8430,0.8263,102.0,37.0,1.0,176.0
Tuned model,0.8703,0.7183,0.9903,0.8327,0.8122,102.0,40.0,1.0,173.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,0.8340,1.1982,0.9000,0.8594,0.7568,1.0000,0.8615,0.7500,27.0,9.0,0.0,28.0
1,2,1.2667,2.1655,0.8443,0.8730,0.7241,1.0000,0.8400,0.8095,34.0,8.0,0.0,21.0
2,3,0.8120,1.1386,0.9077,0.8413,0.6538,0.9444,0.7727,0.8000,36.0,9.0,1.0,17.0
3,4,0.7674,1.0837,0.9410,0.8571,0.6400,1.0000,0.7805,0.8085,38.0,9.0,0.0,16.0
4,5,1.0106,1.7924,0.8709,0.9206,0.8000,1.0000,0.8889,0.8837,38.0,5.0,0.0,20.0


Tuned training MAE: 0.4488
Tuned validation MAE: 0.9381


## 5 Compare scenarios and save the fitted models

The score difference reflects the information available at different prediction times. Do not present the G1/G2 scenario as an early prediction. Save only the two complete fitted pipelines; tables and findings remain inside the notebook.

In [10]:
display(pd.DataFrame({
    "Without G1/G2": tuned_folds_no_grades[METRICS].mean(),
    "With G1/G2": tuned_folds_with_grades[METRICS].mean(),
}).T.round(4))

model_dir = PROJECT_ROOT / "models/experiments"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_no_grades, model_dir / "random_forest__no_prior_grades.joblib")
joblib.dump(best_with_grades, model_dir / "random_forest__with_prior_grades.joblib")
print("Saved fitted pipelines. No experiment tables or reports are exported.")

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Without G1/G2,2.9641,3.9366,0.2081,0.6932,0.5333,0.4939,0.5050,0.7950
With G1/G2,0.9381,1.4757,0.8928,0.8703,0.7149,0.9889,0.8287,0.8104


Saved fitted pipelines. No experiment tables or reports are exported.
